In [1]:
import openai
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\inter\AppData\Local\Temp\ipykernel_9000\4240582308.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print(api_key is not None)

True


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key = api_key , base_url="https://openrouter.ai/api/v1")

In [4]:
llm.invoke("Explain EDA in just 2 lines")

AIMessage(content='EDA, or Exploratory Data Analysis, is the process of exploring and analyzing data to summarize its main characteristics, often using visual methods to identify patterns, trends, and outliers. It helps to understand the structure of the data and make informed decisions when building models or drawing conclusions.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 16, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 9.2e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 9.2e-05, 'upstream_inference_prompt_cost': 8e-06, 'upstream_inference_completions_cost': 8.4e-05}}, 'model_pro

In [5]:
pdf_reader = PyPDFLoader("C:/Users/inter/OneDrive/Documents/GENAI/Research RAG Agent/Rag_researchpaper.pdf")
documents = pdf_reader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunks = text_splitter.split_documents(documents)

RecursiveCharacterTextSplitter splits texts into chunks trying to keep the paragraph together and
avoid losing context

In [6]:
from langchain_community.vectorstores import FAISS


In [7]:
#create the Embeddings
embeddings = OpenAIEmbeddings(api_key=api_key ,  base_url="https://openrouter.ai/api/v1")
db = FAISS.from_documents(documents=chunks, embedding=embeddings)

#FAISS - facebook ai Similarity search --> powerful library for similarity
#  search and clustering the dense vectors

In [8]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate

CONDENSED_QUESTION_PROMPT = PromptTemplate.from_template(
    """Given the following conversation and a follow up question,rephrase the follow up question 
    to be a standalone question

Chat History:
{chat_history}
Follow up Input: {question}
Standalone question:""")

qa = ConversationalRetrievalChain.from_llm(llm=llm, retriever=db.as_retriever() 
                                           , condense_question_llm=CONDENSED_QUESTION_PROMPT ,
                                           return_source_documents = True , verbose = False)

In [9]:
### Ask a query

In [11]:
chat_history = []
query = """What is this research paper all about, explain in just 3-4 lines"""
result = qa({"question":query,"chat_history":chat_history})
print(result["answer"])

C:\Users\inter\AppData\Local\Temp\ipykernel_9000\3212005269.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa({"question":query,"chat_history":chat_history})


The research paper discusses a multi-paragraph reading comprehension model that achieves high accuracy in classifying claims as true or false with minimal evidence. It compares its performance to a RoBERTa model, showing promising results with only claim input for retrieval. Additionally, it analyzes the overlap between documents retrieved by the model and gold evidence annotations.


In [13]:
chat_history = []
query = """What is a rag_sequence model"""
result = qa({"question":query,"chat_history":chat_history})
print(result["answer"])

A RAG-Sequence model is a model that uses the same retrieved document to generate the complete sequence. It treats the retrieved document as a single latent variable that is marginalized to get the sequence-to-sequence probability \(p(y|x)\) via a top-K approximation.
